### Init

In [1]:
import sys

sys.path.append("../src/")

In [2]:
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS, GLOBAL_COMBINED_STATS
import hydra
from hydra.utils import instantiate
from pathlib import Path
import os
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import logging

from utils.data_utils import (
    get_wet_mask,
    get_train_test_ranges,
    gen_data_in_test,
    gen_data_out_test,
    data_CNN_Lateral,
    data_CNN_Dynamic,
    gen_3D_data,
    gen_data_025_lateral,
    gen_data_global_new,
)
from utils.eval_utils import (
    generate_model_rollout,
    compute_mean,
    compute_var,
    compute_corrs_area,
    compute_rmse,
    compute_corrs,
    compute_KE,
    compute_time_spec,
    compute_ACC,
    compute_nino34,
    compute_amo,
    gen_KE_spectrum,
    gen_KE,
    gen_KE_range,
    gen_value_range,
    gen_enstrophy_spectrum,
    gen_enstrophy,
    compute_corrs_single,
    compute_ACC_single,
    compute_RMSE_single,
    compute_mean_single,
)
from utils.subgrid_utils import get_area_tensor
from utils.climate_utils import compute_laplacian_wet
from utils.plot_utils import (
    plot_short_time_stats,
    plot_long_time_stats,
    plot_map,
    plot_error_map,
    plot_both_error_map,
    plot_metrics_KE_spectrum,
    plot_metrics_KE,
    plot_metrics_enstrophy_spectrum,
    plot_metrics_entrophy,
    plot_metrics_corr,
    plot_metrics_rmse,
    plot_metrics_acc,
    plot_metrics_mean,
    plot_metrics_pdf,
    get_initial_snapshot_fig,
    plot_region_based_metric,
    plot_diff_map,
)

import numpy as np
import torch
import xarray as xr
import copy

from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
import copy
from datetime import datetime
import os

In [3]:
def gen_data_global_new(input_vars, extra_vars, output_vars, lag, run_type=""):
    var_dict = {
        "u": "u",
        "v": "v",
        "T": "T",
        "tau_u": "tau_u",
        "tau_v": "tau_v",
        "t_ref": "t_ref",
    }
    if run_type != "":
        run_type = "_" + run_type
    data = xr.open_zarr(
        "/scratch/as15415/Data/Emulation_Data/Global_Ocean_1deg"
        + run_type
        + "_New.zarr"
    )

    data_atmos = xr.open_zarr(
        "/scratch/as15415/Data/Emulation_Data/Data_Atmos_1deg" + run_type + "_New.zarr"
    )
    data_atmos = data_atmos.rename_dims({"lat": "yt_ocean", "lon": "xt_ocean"})
    data_atmos = data_atmos.rename({"lat": "yt_ocean", "lon": "xt_ocean"})

    data = data.sel(time=slice(data_atmos.time[0], data_atmos.time[-1]))
    data_atmos = data_atmos.sel(time=slice(data.time[0], data.time[-1]))

    data_atmos["xu_ocean"] = data.xt_ocean.data
    data_atmos["yu_ocean"] = data.yt_ocean.data

    data = xr.merge([data, data_atmos])

    inputs = []
    extra_in = []
    outputs = []

    for var in input_vars:
        inputs.append(data[var_dict[var]])

    for var in extra_vars:
        extra_in.append(data[var_dict[var]])

    for var in output_vars:
        outputs.append(data[var_dict[var]][lag:])

    inputs = tuple(inputs)
    extra_in = tuple(extra_in)
    outputs = tuple(outputs)

    return inputs, extra_in, outputs

In [4]:
class Eval:
    def __init__(self, args):
        # Getting input, extra input and output
        self.inputs = INPT_VARS[args.exp_num_in]
        self.extra_in = EXTRA_VARS[args.exp_num_extra]
        self.outputs = OUT_VARS[args.exp_num_out]

        self.str_in = "".join([i + "_" for i in self.inputs])
        self.str_ext = "".join([i + "_" for i in self.extra_in])
        self.str_out = "".join([i + "_" for i in self.outputs])

        print("inputs: " + self.str_in)
        print("extra inputs: " + self.str_ext)
        print("outputs: " + self.str_out)

        self.N_atm = len(self.extra_in)  # Number of atmosphere variables
        self.N_in = len(self.inputs)
        if args.lateral:
            self.N_extra = (
                self.N_atm + self.N_in
            )  # Number of atmosphere variables + Lateral boundary variables
        else:
            self.N_extra = self.N_atm  # Number of atmosphere variables
        self.N_out = len(self.outputs)

        self.num_in = int((args.hist + 1) * self.N_in + self.N_extra)

        print("Number of inputs: ", self.num_in)  # 3 (ocean speeds + ocean temp)(t) +
        # 3 (atm wind stresses + atm temp)(t) +
        # 3 (boundary ocean speeds + boundary ocean temp)(t) -> 3 (ocean speeds + ocean temp)(t+1)
        print("Number of outputs: ", self.N_out)  # 3

        # Post-fix strings
        self.str_train = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_out
            + "N_train_4000"
            + "_Lateral_Data_025_no_smooth"
        )
        self.str_save = (
            "steps_"
            + str(args.steps)
            + "_"
            + args.train_region
            + "_"
            + args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )
        self.post_model_name = (
            "Train_"
            + args.train_region
            + "_Test_"
            + args.region
            + "_Test_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "_out"
            + self.str_in
            + "N_train_"
            + str(args.N_samples)
            + "_Lateral_Data_025_no_smooth"
        )
        self.post_pred_name = (
            args.region
            + "_in_"
            + self.str_in
            + "ext_"
            + self.str_ext
            + "N_samples_"
            + str(args.N_samples)
        )

        # Getting start and end indices of train and test
        s_train, e_train, e_test = get_train_test_ranges(
            args.N_samples, args.N_val, args.lag, args.hist, args.interval
        )

        # Saving data
        print("Getting inputs")
        if "global_1" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(
                self.inputs, self.extra_in, self.outputs, args.lag
            )
            total_steps = 366
        elif "global_2x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(
                self.inputs, self.extra_in, self.outputs, args.lag, run_type="2x"
            )
            total_steps = 366
        elif "global_4x" == args.region:
            inputs, extra_in, outputs = gen_data_global_new(
                self.inputs, self.extra_in, self.outputs, args.lag, run_type="4x"
            )
            total_steps = 366
        elif "global_3D":
            inputs, extra_in, outputs = gen_3D_data(
                self.inputs,
                self.extra_in,
                self.outputs,
                args.lag,
                depth_mode=args.depth_mode,
            )
            total_steps = 73
        else:
            raise NotImplementedError

        print("Calculating mask tensors")
        self.wet, self.wet_nan = get_wet_mask(inputs, "cpu")
        self.wet_bool = np.array(self.wet.cpu()).astype(bool)
        wet_lap = compute_laplacian_wet(self.wet_nan, 4)  # hardcoded
        wet_lap = xr.where(wet_lap == 0, 1, np.nan)
        self.wet_lap = np.nan_to_num(wet_lap)
        print("Wet resolution:", self.wet.shape)

        self.time_vec = inputs[0].time.data

        self.time_test = self.time_vec[e_test : (e_test + args.lag * args.N_test)]

        print("Loading Train data")
        train_data = torch.load(
            Path(args.data_dir) / "train_data_cnn_{0}.pt".format(self.str_train),
            map_location=torch.device("cpu"),
        )
        self.train_data = train_data

        if args.save_test_data:
            print("Saving data")
            data_in_test = gen_data_in_test(
                0, e_test, args.N_test, args.lag, args.hist, inputs, extra_in
            )
            data_out_test = gen_data_out_test(
                0, e_test, args.N_test, args.lag, args.hist, outputs
            )
            if "global" in args.region:
                norm_vals = train_data.norm_vals
                if "combined" in args.train_region:
                    assert len(norm_vals) == len(GLOBAL_COMBINED_STATS) and all(
                        np.array_equal(norm_vals[k], GLOBAL_COMBINED_STATS[k])
                        for k in norm_vals
                    )
                self.test_data = data_CNN_Dynamic(
                    data_in_test,
                    data_out_test,
                    self.wet.to(device="cpu"),
                    norm_vals,
                    device=args.device,
                )
                # del train_data
            else:
                raise NotImplementedError()
            torch.save(
                self.test_data,
                Path(args.data_dir) / "test_data_cnn_{0}.pt".format(self.str_save),
            )

        else:
            print("Loading test data")
            self.test_data = torch.load(
                Path(args.data_dir) / "test_data_cnn_{0}.pt".format(self.str_save)
            )

        # Model
        print("Loading model " + args.network)
        if "swin" in args.network.lower():
            model = instantiate(
                args.swin,
                in_channels=self.num_in,
                output_channels=self.N_in,
                pretrain_img_size=[*self.test_data[0][0].shape[1:]],
                wet=self.wet.cuda(),
            )
        elif "unet" in args.network.lower():
            model = instantiate(args.unet, wet=self.wet.cuda())

        full_model_path = args.ckpt_path
        self.full_model_name = args.network + "_" + self.post_model_name
        self.output_channels = model.output_channels

        model = model.to(args.device)
        self.ckpt_path = args.ckpt_path
        self.model = model

        # Stats
        self.mean_out = self.test_data.norm_vals["m_out"]
        self.std_out = self.test_data.norm_vals["s_out"]
        self.mean_in = self.test_data.norm_vals["m_in"]
        self.std_in = self.test_data.norm_vals["s_in"]

        # clim
        self.clim = None
        if args.save_clim_data:
            print("Saving clim")
            clim = np.zeros((total_steps, *self.wet.shape, self.N_out))
            for i in range(self.N_out):
                clim[:, :, :, i] = (
                    outputs[i].groupby("time.dayofyear").mean("time").data
                )
            torch.save(
                clim,
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save),
            )

        else:
            print("Loading clim")
            clim = torch.load(
                Path(args.data_dir) / "clim_cnn_{0}.pt".format(self.str_save)
            )

        self.clim = clim

        # Getting area tensor
        print("Computing area tensor")
        self.grids = xr.open_dataset(
            "/scratch/as15415/Data/CM2x_grids/Grid_New.nc"
        ).rename({"dx": "dxu", "dy": "dyu"})

        self.area = torch.from_numpy(self.grids["area_C"].to_numpy()).to(device="cpu")
        self.dx = self.grids["dxu"].to_numpy()
        self.dy = self.grids["dyu"].to_numpy()

        self.pred_model_path = Path(args.path_dir) / self.full_model_name
        if not os.path.isdir(self.pred_model_path):
            os.makedirs(self.pred_model_path)

        self.Nb = args.Nb
        self.hist = args.hist
        self.lag = args.lag
        self.N_test = args.N_test
        self.N_samples = args.N_samples
        self.output_dir = args.output_dir
        self.region = args.region
        self.steps = args.steps
        self.network = args.model_name_replace
        self.inputs = inputs

        self.pred_region = args.region
        self.pred_names = args.pred_names if args.pred_names else []
        self.pred_paths = args.pred_paths if args.pred_paths else []

        self.JUPYTER_MODE = False

    def send_data_to_cpu(self):
        self.test_data.set_device(device="cpu")

## 2D Fields

### G1, G1

In [5]:
# G1, G1
# ConvNext UNet
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global",
        overrides=[
            "output_dir=./temp/{0}_conv_multiseed_g1g1".format(
                str(datetime.now())[:10]
            ),
            "model_name_replace=ConvNext UNet",
            "train_region=global_1",
            "region=global_1",
            "run_gen_pred=False",  # Multi-Seed Generation
            "network=ConvNext UNet Train1Eval1",
            "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed100/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed200/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=null",
            "pred_paths=null",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

# Adam UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_adamunet_global", overrides=[
#         "output_dir=./temp/{0}_adam_multiseed_g1g1".format(str(datetime.now())[:10]),
#         "model_name_replace=Adam UNet",
#         "train_region=global_1",
#         "region=global_1",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Adam UNet Train1Eval1",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1/adamunetseed/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed100/daam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# Swin
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_swin_global", overrides=[
#         "output_dir=./temp/{0}_swin_multiseed_g1g1".format(str(datetime.now())[:10]),
#         "model_name_replace=Swin",
#         "train_region=global_1",
#         "region=global_1",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Swin Train1Eval1",
#         "swin.embed_dim=60",
#         "exp/modules/blocks@swin.up_sampling_block=transposed_conv_upsample",
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_swintrans60_global_1/swintrans60/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed100/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed200/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=['UNet (Baseline)', 'ConvNext UNet']",
#         "pred_paths=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/Adam UNet Train1Eval1_Train_global_1_Test_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train1Eval1_Train_global_1_Test_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth]"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/state/partition1/job-48180493/ipykernel_666508/3871464725.py:22: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat": "yt_ocean", "lon": "xt_ocean"})
/state/partition1/job-48180493/ipykernel_666508/3871464725.py:22: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat": "yt_ocean", "lon": "xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Train1Eval1
Loading clim
Computing area tensor


### G1, G2x

In [4]:
# ConvNext UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_conv_multiseed_g1g2x".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet",
#         "train_region=global_1",
#         "region=global_2x",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=ConvNext UNet Train1Eval2x",
# "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#             /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed100/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#             /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed200/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# Adam UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_adamunet_global", overrides=[
#         "output_dir=./temp/{0}_adam_multiseed_g1g2x".format(str(datetime.now())[:10]),
#         "model_name_replace=Adam UNet",
#         "train_region=global_1",
#         "region=global_2x",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Adam UNet Train1Eval2x",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1/adamunetseed/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed100/daam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# Swin
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_swin_global", overrides=[
#         "output_dir=./temp/{0}_swin_multiseed_g1g2x".format(str(datetime.now())[:10]),
#         "model_name_replace=Swin",
#         "train_region=global_1",
#         "region=global_2x",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Swin Train1Eval2x",
#         "swin.embed_dim=60",
#         "exp/modules/blocks@swin.up_sampling_block=transposed_conv_upsample",
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_swintrans60_global_1/swintrans60/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed100/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_seed200/swin/saved_nets/swin_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=['UNet (Baseline)', 'ConvNext UNet']",
#         "pred_paths=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/Adam UNet Train1Eval2x_Train_global_1_Test_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train1Eval2x_Train_global_1_Test_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth]"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model Swin Train1Eval2x


/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading clim
Computing area tensor


### G1_2x, G_4x

In [4]:
# ConvNext UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_conv_multiseed_g12x_g4x".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet",
#         "train_region=combined_global_1",
#         "region=global_4x",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "network=ConvNext UNet Train12xEval4x",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_convnextunet_global_1_2x/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_2x_seed100/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_2x_seed200/convnext/saved_nets/convnextunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# Adam UNet
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_adamunet_global", overrides=[
#         "output_dir=./temp/{0}_adam_multiseed_g12x_g4x".format(str(datetime.now())[:10]),
#         "model_name_replace=Adam UNet",
#         "train_region=combined_global_1",
#         "region=global_4x",
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=Adam UNet Train12xEval4x",
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_adamunet_global_1_2x/adam/saved_nets/adamunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_2x_seed100/adam/saved_nets/adamunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_adamunet_global_1_2x_seed200/adam/saved_nets/adamunet_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# Swin
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_swin_global",
        overrides=[
            "output_dir=./temp/{0}_swin_multiseed_g12x_g4x".format(
                str(datetime.now())[:10]
            ),
            "model_name_replace=Swin",
            "train_region=combined_global_1",
            "region=global_4x",
            "N_samples=0",
            "N_val=0",
            "N_test=2000",
            "run_gen_pred=False",  # Multi-Seed Generation
            "network=Swin Train12xEval4x",
            "swin.embed_dim=60",
            "exp/modules/blocks@swin.up_sampling_block=transposed_conv_upsample",
            "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-13-foundation_train_swin_global_1_2x/foundationswin/saved_nets/swin_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_2x_seed100/swin/saved_nets/swin_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-21-foundation_train_swin_global_1_2x_seed200/swin/saved_nets/swin_best_steps_4_global_1_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=['UNet (Baseline)', 'ConvNext UNet']",
            "pred_paths=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/Adam UNet Train12xEval4x_Train_combined_global_1_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth,\
                    /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train12xEval4x_Train_combined_global_1_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth]",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model Swin Train12xEval4x


/ext3/miniconda3/envs/emulator/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading clim
Computing area tensor


### G1 (7k), G4x Test - ConvNext

In [36]:
# ConvNext UNet - 7k
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_conv_multiseed_g17kg4x".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet",
#         "train_region=global_1",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=False", # Multi-Seed Generation
#         "network=ConvNext UNet Train17kEval4x",
#         "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-22-foundation_train_convnextunet_global_1_7k_seed10/cn/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_7000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-22-foundation_train_convnextunet_global_1_7k_seed100/cn/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_7000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-22-foundation_train_convnextunet_global_1_7k_seed200/cn/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_7000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# ConvNext UNet - 4k
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global",
        overrides=[
            "output_dir=./temp/{0}_conv_multiseed_g1g4x".format(
                str(datetime.now())[:10]
            ),
            "model_name_replace=ConvNext UNet 4k",
            "train_region=global_1",
            "region=global_4x",
            "save_test_data=False",  # Done Generation
            "save_clim_data=False",  # Done Generation
            "N_samples=0",
            "N_val=0",
            "N_test=2000",
            "run_gen_pred=True",  # Multi-Seed Generation
            "network=ConvNext UNet Train1Eval4x",
            "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-11-foundation_train_convnextunet_global_1/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed100/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_1_seed200/next/saved_nets/convnextunet_best_steps_4_global_1_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=['ConvNext UNet 7k']",
            "pred_paths=['/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train17kEval4x_Train_global_1_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth']",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)


e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
Calculating mask tensors


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Train1Eval4x
Loading clim
Computing area tensor


### G1 Preloaded, G2x Trained, G4x Test - ConvNext

#### Old

In [ ]:
# 50%
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_convnext_blends".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet 50%",
#         "network=ConvNext UNet Preload1Train2xEval4x50",
#         "train_region=global_2x",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=False", # Done Generation
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_50p/cnext/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# 10%
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_convnext_blends".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet 10%",
#         "network=ConvNext UNet Preload1Train2xEval4x10",
#         "train_region=global_2x",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=False", # Done Generation
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_10p/connext/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

#### Use

In [10]:
# 25%
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_convnext_blends".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet 25%",
#         "network=ConvNext UNet Preload1Train2xEval4x25",
#         "train_region=global_2x",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=False", # Done Generation
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_25p/cnext/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_25p_seed100/per/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_25p_seed200/200/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# 5%
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_convnext_blends".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet 5%",
#         "network=ConvNext UNet Preload1Train2xEval4x05",
#         "train_region=global_2x",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=False", # Done Generation
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_05p/percent/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_05p_seed100/5/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_05p_seed200/5/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# 1%
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global",
        overrides=[
            "output_dir=./temp/{0}_convnext_blends_1p_seeds".format(
                str(datetime.now())[:10]
            ),
            "model_name_replace=1% 2x CO2",
            "network=ConvNext UNet Preload1Train2xEval4x01",
            "train_region=global_2x",
            "region=global_4x",
            "save_test_data=False",  # Done Generation
            "save_clim_data=False",  # Done Generation
            "N_samples=0",
            "N_val=0",
            "N_test=2000",
            "run_gen_pred=False",  # Done Generation
            "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_01p/1p/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_01p_seed100/1/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_01p_seed200/1/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=['0% 2x CO2', '25% 2x CO2', '5% 2x CO2']",
            "pred_paths=['/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train1Eval4x_Train_global_1_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth',\
                    '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Preload1Train2xEval4x25_Train_global_2x_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth',\
                    '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Preload1Train2xEval4x05_Train_global_2x_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth']",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
Calculating mask tensors


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Preload1Train2xEval4x01
Loading clim
Computing area tensor


#### Fixed

In [11]:
# 25%
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_convnext_blends_fixed".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet 25%",
#         "network=ConvNext UNet Preload1Train2xEval4x_Fixed_25",
#         "train_region=global_2x",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=True", # Done Generation
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_25p/cnext/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_25p_seed100/per/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_25p_seed200/200/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# 5%
# with initialize_config_dir(version_base=None, config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs"):
#     args = compose(config_name="exp/eval_unet_global", overrides=[
#         "output_dir=./temp/{0}_convnext_blends_fixed".format(str(datetime.now())[:10]),
#         "model_name_replace=ConvNext UNet 5%",
#         "network=ConvNext UNet Preload1Train2xEval4x_Fixed_05",
#         "train_region=global_2x",
#         "region=global_4x",
#         "save_test_data=False", # Done Generation
#         "save_clim_data=False", # Done Generation
#         "N_samples=0",
#         "N_val=0",
#         "N_test=2000",
#         "run_gen_pred=True", # Done Generation
#         "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_05p/percent/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_05p_seed100/5/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
#                     /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_05p_seed200/5/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
#         "pred_names=null",
#         "pred_paths=null"
#     ])
# if not os.path.exists(args.output_dir):
#     os.mkdir(args.output_dir)

# 1%
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global",
        overrides=[
            "output_dir=./temp/{0}_convnext_blends_fixed_1p_seeds".format(
                str(datetime.now())[:10]
            ),
            "model_name_replace=1% 2x CO2",
            "network=ConvNext UNet Preload1Train2xEval4x_Fixed_01",
            "train_region=global_2x",
            "region=global_4x",
            "save_test_data=False",  # Done Generation
            "save_clim_data=False",  # Done Generation
            "N_samples=0",
            "N_val=0",
            "N_test=2000",
            "run_gen_pred=True",  # Done Generation
            "ckpt_path=[/scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_01p/1p/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_01p_seed100/1/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-23-foundation_train_convnextunet_loadedglobal1_global_2x_01p_seed200/1/saved_nets/convnextunet_epoch_100_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=['0% 2x CO2', '25% 2x CO2', '5% 2x CO2']",
            "pred_paths=['/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train1Eval4x_Train_global_1_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth',\
                    '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Preload1Train2xEval4x_Fixed_25_Train_global_2x_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth',\
                    '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Preload1Train2xEval4x_Fixed_05_Train_global_2x_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth']",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lat' to 'yt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1049: UserWarning: rename 'lon' to 'xt_ocean' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat":"yt_ocean","lon":"xt_ocean"})


Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Preload1Train2xEval4x_Fixed_01
Loading clim
Computing area tensor


### G2

In [5]:
# G2x G4x
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global",
        overrides=[
            "output_dir=./temp/{0}_convnext_g2x_g4x".format(str(datetime.now())[:10]),
            "model_name_replace=2x CO2",
            "network=ConvNext UNet Train2xEval4x",
            "train_region=global_2x",
            "region=global_4x",
            "save_test_data=False",  # Done Generation
            "save_clim_data=False",  # Done Generation
            "N_samples=0",
            "N_val=0",
            "N_test=2000",
            "run_gen_pred=False",  # Done Generation
            "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-05-13-foundation_train_convnextunet_global_2x/conv/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_2x_seed100/conv/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt,\
                    /scratch/sg7761/m2lines/Ocean_Emulator/train/2024-05-21-foundation_train_convnextunet_global_2x_seed200/conv/saved_nets/convnextunet_best_steps_4_global_2x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=['0% 2x CO2', '1% 2x CO2', '5% 2x CO2']",
            "pred_paths=['/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train1Eval4x_Train_global_1_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth',\
                    '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Preload1Train2xEval4x01_Train_global_2x_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth',\
                    '/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Preload1Train2xEval4x05_Train_global_2x_Test_global_4x_Test_in_u_v_T_ext_tau_u_tau_v_t_ref__outu_v_T_N_train_0_Lateral_Data_025_no_smooth']",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

e = Eval(args)

inputs: u_v_T_
extra inputs: tau_u_tau_v_t_ref_
outputs: u_v_T_
Number of inputs:  6
Number of outputs:  3
Getting inputs
<xarray.Dataset>
Dimensions:   (time: 4748, yt_ocean: 180, xt_ocean: 360)
Coordinates:
  * time      (time) object 0190-01-01 12:00:00 ... 0202-12-31 12:00:00
  * xt_ocean  (xt_ocean) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * yt_ocean  (yt_ocean) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Data variables:
    T         (time, yt_ocean, xt_ocean) float64 dask.array<chunksize=(180, 180, 10), meta=np.ndarray>
    u         (time, yt_ocean, xt_ocean) float64 dask.array<chunksize=(180, 180, 10), meta=np.ndarray>
    v         (time, yt_ocean, xt_ocean) float64 dask.array<chunksize=(180, 180, 10), meta=np.ndarray>


/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1287: UserWarning: rename 'lat' to 'y' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat": "y", "lon": "x"})
/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/notebooks/../src/utils/data_utils.py:1287: UserWarning: rename 'lon' to 'x' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  data_atmos = data_atmos.rename({"lat": "y", "lon": "x"})


AttributeError: 'Dataset' object has no attribute 'x'

In [6]:
# G3D Surface
with initialize_config_dir(
    version_base=None,
    config_dir="/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/configs",
):
    args = compose(
        config_name="exp/eval_unet_global_3D",
        overrides=[
            "output_dir=./temp/{0}_convnext_3D_surface".format(
                str(datetime.now())[:10]
            ),
            "model_name_replace=Convnext 3D Surface",
            "network=ConvNext UNet Train3DSurfaceEval3DSurface",
            "data_dir=/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/save_data/2024-06-16-save_3D_data_surface_test/surface/data",
            "train_region=global_3D",
            "region=global_3D",
            "depth_mode=surface",
            "save_test_data=False",  # Done Generation
            "save_clim_data=False",  # Done Generation
            "N_samples=4000",
            "N_val=140",
            "N_test=600",
            "run_gen_pred=False",  # Done Generation
            "ckpt_path=[/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/train/2024-06-16-train_convnextunet_global_3D_surface/surface3/saved_nets/convnextunet_best_steps_4_global_3D_Test_in_uo_vo_thetao_so_zos_ext_tauuo_tauvo_hfds__outuo_vo_thetao_so_zos_N_train_4000_Lateral_Data_025_no_smooth.pt]",
            "pred_names=null",
            "pred_paths=null",
        ],
    )
if not os.path.exists(args.output_dir):
    os.mkdir(args.output_dir)

e = Eval(args)

inputs: uo_vo_thetao_so_zos_
extra inputs: tauuo_tauvo_hfds_
outputs: uo_vo_thetao_so_zos_
Number of inputs:  8
Number of outputs:  5
Getting inputs
Calculating mask tensors
Wet resolution: torch.Size([180, 360])
Loading Train data
Loading test data
Loading model ConvNext UNet Train3DSurfaceEval3DSurface
Loading clim
Computing area tensor


### HuggingFace

In [7]:
model_path = args.ckpt_path[0]
e.model.load_state_dict(torch.load(model_path, map_location=torch.device("cuda")))

<All keys matched successfully>

In [8]:
e.model.push_to_hub("M2LInES/ocean_surface_emulation")

model.safetensors:   0%|          | 0.00/63.6M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/M2LInES/ocean_surface_emulation/commit/f562f05a555693f999dc993499d91f606716ce71', commit_message='Push model using huggingface_hub.', commit_description='', oid='f562f05a555693f999dc993499d91f606716ce71', pr_url=None, pr_revision=None, pr_num=None)

In [9]:
e.model.from_pretrained("M2LInES/ocean_surface_emulation")

config.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]

TypeError: UNet.__init__() missing 8 required positional arguments: 'core_block', 'down_sampling_block', 'up_sampling_block', 'activation', 'ch_width', 'dilation', 'n_layers', and 'wet'